In [1]:
# =============================================================
# HR Predictive Analytics: Synthetic Dataset Generator
# Author: Kevin Cauley
# Purpose: Generate 7 HR Tables for Modeling in RStudio
# Sources: BLS (2024), Gallup (2023), SHRM, Zhong et al. (2023), Wells et al. (2023),
#          Forbes (2023), Reeja et al. (2021), Vitak & Zimmer (2023), Fallucchi et al. (2023)
# =============================================================

import pandas as pd
import numpy as np
import random
from faker import Faker

# Utility Functions
def likert_score(prob_dist):
    return np.random.choice(range(1, len(prob_dist) + 1), p=prob_dist)

def binary_outcome(prob_yes):
    return np.random.choice(["Yes", "No"], p=[prob_yes, 1 - prob_yes])

def count_score(probabilities, values):
    return np.random.choice(values, p=probabilities)

# Setup
np.random.seed(42)
random.seed(42)
Faker.seed(42)
fake = Faker()
num_employees = 1000  # Fallucchi et al. (2023)
employee_ids = [f"E{i:04d}" for i in range(1, num_employees + 1)]
excel_data = {}

# =============================================================
# 1. DATA & DEMOGRAPHICS
# =============================================================

# BLS Salary Bands
bls_salary_ranges = {
    "Software Developer": {"10th": 68130, "25th": 87540, "50th": 113270, "75th": 139800, "90th": 167410},
    "Information Security Analyst": {"10th": 69220, "25th": 90040, "50th": 120370, "75th": 153540, "90th": 182370},
    "Computer Systems Analyst": {"10th": 63230, "25th": 80370, "50th": 103790, "75th": 132580, "90th": 165690},
    "Computer Support Specialist": {"10th": 38500, "25th": 47670, "50th": 60820, "75th": 78540, "90th": 101230},
    "Computer Network Architect": {"10th": 77960, "25th": 100130, "50th": 129830, "75th": 164070, "90th": 195000},
    "Computer Programmer": {"10th": 55380, "25th": 70970, "50th": 99320, "75th": 124770, "90th": 153710},
    "Web Developer": {"10th": 44900, "25th": 59600, "50th": 80570, "75th": 102860, "90th": 129760},
    "CIS Manager": {"10th": 101590, "25th": 131770, "50th": 169520, "75th": 214050, "90th": 250000},
    "SysAdmin": {"10th": 56000, "25th": 71000, "50th": 94000, "75th": 117000, "90th": 140000},
    "QA Analyst": {"10th": 59000, "25th": 72000, "50th": 99600, "75th": 120000, "90th": 145000},
    "Web Designer": {"10th": 42000, "25th": 59000, "50th": 86600, "75th": 105000, "90th": 125000}
}

occupation_salary_map = {
    "Software Developer": "Applications", "Information Security Analyst": "Security",
    "Computer Systems Analyst": "Infrastructure", "Computer Support Specialist": "Support",
    "Computer Network Architect": "Infrastructure", "Computer Programmer": "Applications",
    "Web Developer": "Web", "CIS Manager": "Management", "SysAdmin": "Infrastructure",
    "QA Analyst": "QA", "Web Designer": "Web"
}

def realistic_salary(occupation, tenure_bin):
    p = bls_salary_ranges[occupation]
    if tenure_bin == "0-1": return int(np.random.uniform(p["10th"], p["25th"]))
    elif tenure_bin == "1-3": return int(np.random.uniform(p["25th"], p["50th"]))
    elif tenure_bin == "3-5": return int(np.random.normal((p["50th"] + p["75th"])/2, 0.1*(p["75th"] - p["50th"])))
    elif tenure_bin == "5-10": return int(np.random.normal((p["75th"] + p["90th"])/2, 0.1*(p["90th"] - p["75th"])))
    elif tenure_bin == "10+": return int(np.random.normal(p["90th"], 0.05 * p["90th"]))
    return int(np.random.uniform(60000, 120000))

# Distributions
age_dist = [0.01, 0.05, 0.25, 0.25, 0.20, 0.20, 0.04]
gender_dist = [0.74, 0.26]
race_dist = [0.64, 0.10, 0.23, 0.03]
ethnicity_dist = [0.08, 0.92]

# Generate Demographics
demographics = []
for emp_id in employee_ids:
    occ = random.choice(list(occupation_salary_map.keys()))
    dept = occupation_salary_map[occ]
    tenure_bin = np.random.choice(["0-1", "1-3", "3-5", "5-10", "10+"], p=[0.30, 0.40, 0.15, 0.10, 0.05])
    tenure = round(np.random.uniform({"0-1": 0.1, "1-3": 1.0, "3-5": 3.0, "5-10": 5.0, "10+": 10.0}[tenure_bin], {"0-1": 1.0, "1-3": 3.0, "3-5": 5.0, "5-10": 10.0, "10+": 20.0}[tenure_bin]), 1)
    salary = realistic_salary(occ, tenure_bin)
    seniority = {"0-1": "Entry", "1-3": "Junior", "3-5": "Mid", "5-10": "Senior", "10+": "Lead"}[tenure_bin]
    age_group = np.random.choice(["16-19", "20-24", "25-34", "35-44", "45-54", "55-64", "65+"], p=age_dist)
    gender = np.random.choice(["Male", "Female"], p=gender_dist)
    race = np.random.choice(["White", "Black", "Asian", "Other"], p=race_dist)
    ethnicity = np.random.choice(["Hispanic", "Non-Hispanic"], p=ethnicity_dist)
    work_arrangement = np.random.choice(["Hybrid", "Remote", "On-site"], p=[0.55, 0.35, 0.10] if gender == "Female" and age_group in ["35-44", "45-54", "55-64"] else [0.50, 0.25, 0.25])
    distance = round(np.random.exponential(scale=10), 1)

    demographics.append([emp_id, fake.first_name(), fake.last_name(), occ, dept, salary, tenure, seniority, work_arrangement, distance, age_group, gender, race, ethnicity])

excel_data["Data & Demographics"] = pd.DataFrame(demographics, columns=[
    "Employee_ID", "First_Name", "Last_Name", "Occupation", "Department",
    "Average_Salary", "Tenure_Years", "Seniority", "Work_Arrangement",
    "Distance_From_Home", "Age_Group", "Gender", "Race", "Ethnicity"
])

# =============================================================
# 2. METRICS
# =============================================================

def generate_overtime():
    category = np.random.choice(["low", "med", "high"], p=[0.60, 0.30, 0.10])
    return int(np.random.uniform(0, 50)) if category == "low" else int(np.random.uniform(51, 100)) if category == "med" else int(np.random.uniform(101, 200))

metrics_data = []
for emp_id in employee_ids:
    metrics_data.append([
        emp_id,
        max(0, int(np.random.normal(50, 20))),
        generate_overtime(),
        likert_score([0.05, 0.10, 0.35, 0.35, 0.15]),
        likert_score([0.05, 0.10, 0.30, 0.35, 0.20]),
        likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        likert_score([0.10, 0.20, 0.30, 0.25, 0.15]),
        likert_score([0.05, 0.10, 0.25, 0.35, 0.25]),
        likert_score([0.05, 0.10, 0.25, 0.35, 0.25])
    ])

excel_data["Metrics"] = pd.DataFrame(metrics_data, columns=[
    "Employee_ID", "Actual_Training_Hours", "Overtime_Hours",
    "Performance_Score", "Productivity_Score", "Discretionary_Effort",
    "Engagement_Level", "Organizational_Commitment", "Role_Clarity"
])

# =============================================================
# 3. SATISFACTION
# Sources: Gallup (2023), SHRM, Zhong et al.
# =============================================================
satisfaction_data = []
for emp_id in employee_ids:
    satisfaction_data.append({
        "Employee_ID": emp_id,
        "Job_Satisfaction": likert_score([0.05, 0.10, 0.25, 0.35, 0.25]),
        "Workplace_Environment_Satisfaction": likert_score([0.10, 0.20, 0.30, 0.25, 0.15]),
        "Work_Life_Balance_Satisfaction": likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        "Perceived_Organizational_Support": likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        "Relationship_With_Manager": likert_score([0.05, 0.10, 0.20, 0.35, 0.30]),
        "Relationship_With_Coworkers": likert_score([0.05, 0.10, 0.25, 0.30, 0.30]),
        "Team_Collaboration": likert_score([0.05, 0.10, 0.25, 0.35, 0.25]),
        "Perceived_Career_Growth_Opportunities": likert_score([0.10, 0.20, 0.30, 0.25, 0.15]),
        "Promotion_Opportunity": likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        "Learning_Opportunities": likert_score([0.10, 0.15, 0.30, 0.30, 0.15]),
        "Manager_Support_Score": likert_score([0.05, 0.10, 0.30, 0.30, 0.25]),
        "Perceived_Fairness_Score": likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        "Trust_In_Management": likert_score([0.10, 0.15, 0.25, 0.30, 0.20]),
        "Job_Involvement": likert_score([0.10, 0.20, 0.35, 0.25, 0.10]),
        "Recognition_Score": likert_score([0.10, 0.15, 0.30, 0.25, 0.20]),
        "Voice_In_Decision_Making": likert_score([0.15, 0.20, 0.25, 0.25, 0.15]),
        "Alignment_With_Mission": likert_score([0.10, 0.20, 0.30, 0.25, 0.15]),
        "Manager_Communication_Quality": likert_score([0.05, 0.15, 0.30, 0.30, 0.20]),
        "Feeling_Valued_At_Work": likert_score([0.10, 0.15, 0.30, 0.30, 0.15])
    })

excel_data["Satisfaction"] = pd.DataFrame(satisfaction_data)

# 4. WELLNESS
# Sources: Gallup, SHRM, Reeja et al. (2021), Khalid & Syed (2024)
# =============================================================
wellness_data = []
for emp_id in employee_ids:
    burnout = likert_score([0.05, 0.10, 0.20, 0.25, 0.20, 0.15, 0.05])
    wellbeing = likert_score([0.02, 0.05, 0.10, 0.25, 0.30, 0.20, 0.08])
    anxiety = likert_score([0.10, 0.15, 0.30, 0.25, 0.20])
    depression = likert_score([0.10, 0.20, 0.30, 0.25, 0.15])
    stress = likert_score([0.05, 0.10, 0.25, 0.30, 0.30])
    mental_days = np.random.poisson(1.2 if burnout >= 5 or stress >= 4 else 0.4)
    absenteeism = np.random.poisson(2.2 if anxiety >= 4 or depression >= 4 else 0.6)
    counseling = np.random.choice(["Yes", "No"], p=[0.35, 0.65])

    # ➕ New Research-Backed Variables
    workload_pressure = likert_score([0.10, 0.20, 0.30, 0.25, 0.15])              # SHRM: high pressure in IT
    coping_skills = likert_score([0.05, 0.10, 0.25, 0.30, 0.30])                  # Mental resilience
    peer_support = likert_score([0.05, 0.10, 0.30, 0.30, 0.25])                   # AIoT study
    manager_empathy = likert_score([0.05, 0.10, 0.25, 0.35, 0.25])                # Gallup: empathy & burnout
    leadership_style = np.random.choice(["Supportive", "Autocratic", "Laissez-faire"], p=[0.50, 0.30, 0.20])
    stressors = np.random.poisson(1.5 if workload_pressure >= 4 else 0.7)

    wellness_data.append([
        emp_id, burnout, wellbeing, anxiety, depression, stress,
        mental_days, absenteeism, counseling, workload_pressure,
        coping_skills, peer_support, manager_empathy, leadership_style, stressors
    ])

excel_data["Wellness"] = pd.DataFrame(wellness_data, columns=[
    "Employee_ID", "Burnout_Score", "Wellbeing_Score", "Anxiety_Level",
    "Depression_Level", "Stress_Level", "Mental_Health_Days_Taken",
    "Absenteeism_Due_To_Mental_Health", "Access_To_Counseling",
    "Workload_Pressure", "Coping_Skills", "Peer_Support",
    "Manager_Empathy", "Leadership_Style", "Stressors_Experienced"
])

# =============================================================
# 5. MONITORING
# Sources: Wells et al., Vitak & Zimmer (2023), Forbes, Gallup, Stanford
# =============================================================

monitoring_data = []

for emp_id in employee_ids:
    time_method = np.random.choice(
        ["Clock-in/out", "Timesheets", "Keystroke Monitoring", "Activity Timer", "None"],
        p=[0.20, 0.25, 0.15, 0.25, 0.15])

    surveillance_method = np.random.choice(
        ["Screenshot Capturing", "Webcam Monitoring", "Screen Recording", "Email Scanning", "Browser Logging", "App Usage", "None"],
        p=[0.15, 0.10, 0.15, 0.20, 0.20, 0.15, 0.05])

    tool_count = int(time_method != "None") + int(surveillance_method != "None")
    monitoring_freq = likert_score([0.10, 0.20, 0.35, 0.25, 0.10])
    intrusiveness = likert_score([0.05, 0.10, 0.25, 0.35, 0.25])
    productivity_impact = likert_score([0.20, 0.25, 0.30, 0.15, 0.10])
    satisfaction_impact = likert_score([0.15, 0.20, 0.30, 0.20, 0.15])
    violated = np.random.poisson(1)
    perceived_purpose = np.random.choice(["Developmental", "Deterrent"], p=[0.60, 0.40])
    ai_supervision = np.random.choice(["Yes", "No"], p=[0.30, 0.70])

    monitoring_data.append([
        emp_id,
        time_method,
        surveillance_method,
        tool_count,
        monitoring_freq,
        intrusiveness,
        productivity_impact,
        satisfaction_impact,
        violated,
        perceived_purpose,
        ai_supervision
    ])

excel_data["Monitoring"] = pd.DataFrame(monitoring_data, columns=[
    "Employee_ID", "Time_Tracking_Method_Used", "Surveillance_Method_Used",
    "Total_Monitoring_Tools", "Monitoring_Frequency", "Monitoring_Intrusiveness",
    "Monitoring_Productivity_Impact", "Monitoring_Satisfaction_Impact",
    "Violations_Logged_By_Monitoring_Software", "Perceived_Monitoring_Purpose",
    "AI_Supervision"
])

# =============================================================
# 6. PRIVACY
# Sources: Vitak & Zimmer (2023), Wells et al., Gallup
# =============================================================
privacy_data = []

for emp_id in employee_ids:
    intrusiveness = likert_score([0.05, 0.15, 0.25, 0.30, 0.25])
    complaints = np.random.poisson(0.7 if intrusiveness > 3 else 0.2)

    privacy_data.append([
        emp_id,
        likert_score([0.10, 0.15, 0.25, 0.30, 0.20]),  # Monitoring Transparency
        likert_score([0.05, 0.10, 0.25, 0.30, 0.30]),  # Privacy Concern
        intrusiveness,
        likert_score([0.10, 0.15, 0.25, 0.30, 0.20]),  # Trust in Monitoring
        complaints
    ])

excel_data["Privacy"] = pd.DataFrame(privacy_data, columns=[
    "Employee_ID", "Monitoring_Transparency", "Privacy_Concern_Score",
    "Perceived_Intrusiveness", "Trust_In_Monitoring", "Privacy_Complaints_Score"
])

# =============================================================
# =============================================================
# 7. ATTRITION (Updated with Research-Based Logic)
# Sources: 
#   - Gallup (2023): Burnout, job satisfaction, wellbeing
#   - Zhong et al. (2023): Career growth & quiet quitting
#   - SHRM: Organizational commitment, retention
#   - Fallucchi et al. (2023): Data-driven turnover predictors
#   - Wells et al. (2007): Monitoring purpose (developmental vs deterrent)
#   - Vitak & Zimmer (2023): Intrusiveness, transparency, AI surveillance
#   - Siegel et al. (2022): Meta-analysis on monitoring & stress
# =============================================================

attrition_data = []

# Convert tables to fast-lookup dictionaries
metrics_lookup = excel_data["Metrics"].set_index("Employee_ID").to_dict("index")
satisfaction_lookup = excel_data["Satisfaction"].set_index("Employee_ID").to_dict("index")
wellness_lookup = excel_data["Wellness"].set_index("Employee_ID").to_dict("index")
monitoring_lookup = excel_data["Monitoring"].set_index("Employee_ID").to_dict("index")
demo_lookup = excel_data["Data & Demographics"].set_index("Employee_ID").to_dict("index")

for emp_id in employee_ids:
    metrics = metrics_lookup[emp_id]
    satisfaction = satisfaction_lookup[emp_id]
    wellness = wellness_lookup[emp_id]
    monitoring = monitoring_lookup[emp_id]
    demographics = demo_lookup[emp_id]

    # --- Turnover Intention ---
    # Gallup & SHRM: Low job satisfaction, burnout, low perceived support → high turnover risk
    if (
        satisfaction["Job_Satisfaction"] <= 2 or
        wellness["Burnout_Score"] >= 6 or
        satisfaction["Perceived_Organizational_Support"] <= 2 or
        np.random.choice([0, 1], p=[0.85, 0.15]) == 0  # Simulate no promotion
    ):
        turnover_intention = binary_outcome(0.6)
    elif metrics["Organizational_Commitment"] >= 4 and demographics["Average_Salary"] > 100000:
        turnover_intention = binary_outcome(0.05)
    else:
        turnover_intention = binary_outcome(0.25)

    # --- Quiet Quitting ---
    # Zhong et al. (2023): Burnout, low commitment, poor career outlook → quiet quitting
    if (
        wellness["Burnout_Score"] >= 5 and
        metrics["Organizational_Commitment"] <= 2 and
        satisfaction["Perceived_Career_Growth_Opportunities"] <= 2
    ):
        quiet_quitting = "Yes"
    else:
        quiet_quitting = binary_outcome(0.20)

    # --- Monitoring Turnover Risk (Literature-Backed) ---
    # Based on Wells et al., Vitak & Zimmer, Siegel et al., and AI monitoring concerns
    high_invasiveness = monitoring["Monitoring_Intrusiveness"] >= 4
    low_trust = monitoring.get("Trust_In_Monitoring", 3) <= 2  # AI is Watching You
    low_transparency = monitoring.get("Monitoring_Transparency", 3) <= 2
    ai_based = monitoring["AI_Supervision"] == "Yes"
    punitive_purpose = monitoring["Perceived_Monitoring_Purpose"] == "Deterrent"  # Wells et al.
    high_privacy_concern = monitoring.get("Privacy_Concern_Score", 3) >= 4
    high_burnout = wellness.get("Burnout_Score", 3) >= 6  # Gallup
    low_wellbeing = wellness.get("Wellbeing_Score", 3) <= 2
    poor_impact_satisfaction = monitoring["Monitoring_Satisfaction_Impact"] <= 2

    # Multi-factor turnover risk logic from monitoring design + psychological signals
    if (
        (high_invasiveness and low_trust) or
        (punitive_purpose and poor_impact_satisfaction) or
        (ai_based and high_privacy_concern) or
        (low_transparency and high_burnout)
    ):
        monitoring_turnover_risk = "Yes"
    elif low_wellbeing:
        monitoring_turnover_risk = binary_outcome(0.40)
    else:
        monitoring_turnover_risk = binary_outcome(0.15)

    # Final record for each employee
    attrition_data.append([
        emp_id,
        turnover_intention,
        binary_outcome(0.40),                            # Voluntary Resignation
        binary_outcome(0.05),                            # Involuntary Termination
        quiet_quitting,
        likert_score([0.10, 0.20, 0.30, 0.25, 0.15]),     # Employer Promise Broken
        count_score([0.60, 0.25, 0.10, 0.05], [0, 1, 2, 3]),
        np.random.choice(["No", "Yes"], p=[0.85, 0.15]),  # Promotion Last 12 Months
        monitoring_turnover_risk
    ])

# Export to Excel
excel_data["Attrition"] = pd.DataFrame(attrition_data, columns=[
    "Employee_ID", "Turnover_Intention", "Voluntary_Resignation", "Involuntary_Termination",
    "Quiet_Quitting", "Employer_Promise_Broken", "Internal_Job_Applications",
    "Promotion_Last_12_Months", "Monitoring_Turnover_Risk"
])

# =============================================================
# Export to Excel
# =============================================================
output_file = r"data/raw/HR_Data.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in excel_data.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Successfully Exported to '{output_file}'")

Successfully Exported to 'data/raw/HR_Data.xlsx'
